In [1]:
# Spotify Music Clustering in Google Colab

# Step 0: Install and import necessary libraries
!pip install kagglehub scikit-learn plotly

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px
import kagglehub

# Step 1: Download dataset directly from Kaggle
path = kagglehub.dataset_download("zaheenhamidani/ultimate-spotify-tracks-db")
print("Path to dataset files:", path)

# Step 2: Load SpotifyFeatures.csv
df = pd.read_csv(f"{path}/SpotifyFeatures.csv")

# Step 3: Sample the dataset for faster computation (e.g., 20k tracks)
sample_df = df.sample(n=20000, random_state=42)

# Step 4: Select audio features for clustering
features = ['danceability', 'energy', 'tempo', 'loudness', 'valence']
X = sample_df[features]

# Step 5: Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 6: Apply KMeans clustering
k = 5  # choose number of clusters (can be tuned)
kmeans = KMeans(n_clusters=k, random_state=42)
sample_df['cluster'] = kmeans.fit_predict(X_scaled)

# Step 7: PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
sample_df['pca1'] = X_pca[:,0]
sample_df['pca2'] = X_pca[:,1]

# Step 8: t-SNE for another visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=50, n_iter=500)
X_tsne = tsne.fit_transform(X_scaled)
sample_df['tsne1'] = X_tsne[:,0]
sample_df['tsne2'] = X_tsne[:,1]

# Step 9: Visualization with Plotly
# PCA plot
fig_pca = px.scatter(sample_df, x='pca1', y='pca2', color='cluster',
                     hover_data=['track_name', 'artist_name', 'genre'],
                     title="Spotify Songs Clustering (PCA)")
fig_pca.show()

# t-SNE plot
fig_tsne = px.scatter(sample_df, x='tsne1', y='tsne2', color='cluster',
                      hover_data=['track_name', 'artist_name', 'genre'],
                      title="Spotify Songs Clustering (t-SNE)")
fig_tsne.show()

# Step 10: Insights
for i in range(k):
    cluster_songs = sample_df[sample_df['cluster']==i]
    print(f"\nCluster {i} summary:")
    print(f"Average Danceability: {cluster_songs['danceability'].mean():.2f}")
    print(f"Average Energy: {cluster_songs['energy'].mean():.2f}")
    print(f"Average Valence: {cluster_songs['valence'].mean():.2f}")
    print(f"Top genres in this cluster: {cluster_songs['genre'].value_counts().head(3).to_dict()}")

Using Colab cache for faster access to the 'ultimate-spotify-tracks-db' dataset.
Path to dataset files: /kaggle/input/ultimate-spotify-tracks-db


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(



Cluster 0 summary:
Average Danceability: 0.51
Average Energy: 0.76
Average Valence: 0.51
Top genres in this cluster: {'Ska': 296, 'Anime': 267, 'Children’s Music': 257}

Cluster 1 summary:
Average Danceability: 0.56
Average Energy: 0.37
Average Valence: 0.33
Top genres in this cluster: {'Folk': 368, 'Jazz': 302, 'Indie': 284}

Cluster 2 summary:
Average Danceability: 0.27
Average Energy: 0.14
Average Valence: 0.13
Top genres in this cluster: {'Soundtrack': 640, 'Classical': 624, 'Opera': 589}

Cluster 3 summary:
Average Danceability: 0.73
Average Energy: 0.67
Average Valence: 0.74
Top genres in this cluster: {'Reggaeton': 481, 'Reggae': 474, 'Hip-Hop': 352}

Cluster 4 summary:
Average Danceability: 0.54
Average Energy: 0.75
Average Valence: 0.37
Top genres in this cluster: {'Comedy': 416, 'Alternative': 278, 'Children’s Music': 271}


In [3]:
# Save model artifacts
import pickle

# Save the scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save the trained KMeans model
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

# Save the sample dataframe for cluster insights (with all necessary info)
sample_df[['cluster', 'genre', 'danceability', 'energy', 'valence', 'loudness', 'tempo',
           'track_name', 'artist_name', 'popularity']].to_csv('spotify_sample.csv', index=False)